[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/10_gqa_solution.ipynb)

# 🔴 Solution: Grouped Query Attention

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `10_gqa.ipynb` first.

---
Implement **grouped-query attention** (GQA), the attention variant used by
Llama-2 70B, Llama-3, Mistral and most current open models.

Query heads are split into groups, and each **group shares one key/value head**.

### Signature
```python
class GroupQueryAttention(nnx.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, *, rngs: nnx.Rngs): ...
    def __call__(self, x): ...
```

### Requirements
- `self.W_q`: `nnx.Linear(d_model, d_model)`
- `self.W_k`, `self.W_v`: `nnx.Linear(d_model, num_kv_heads * d_k)` — **smaller**
- `self.W_o`: `nnx.Linear(d_model, d_model)`
- `self.d_k = d_model // num_heads`
- `num_heads` must be divisible by `num_kv_heads`

### The two endpoints
- `num_kv_heads == num_heads` → ordinary multi-head attention
- `num_kv_heads == 1` → multi-query attention (MQA)

GQA is the middle: it recovers most of MQA's savings without MQA's quality
drop.

### Why it exists — the KV cache, not the FLOPs
The projections get slightly cheaper, but that is not the motivation. During
generation the KV cache holds
$2 \times B \times H_{kv} \times seq \times d_k$ values, and at long context it
dwarfs the weights. Cutting $H_{kv}$ from 64 to 8 cuts the cache **8×**, which
is the difference between fitting a long-context batch in memory and not.

Llama-2 70B uses 64 query heads and 8 KV heads. Quality is within noise of full
MHA; the cache is one eighth the size.

### The trap: repeat vs tile
Query head $i$ must pair with KV head $i \,/\, \text{repeats}$ (integer
division), so the KV heads need each entry duplicated **in place**:
`[a, b] -> [a, a, a, a, b, b, b, b]`. Tiling gives `[a, b, a, b, a, b, a, b]`
instead, which silently pairs the wrong heads — shapes match, gradients flow,
and the model just learns worse. `jnp.repeat(k, repeats, axis=1)` is the
correct spelling.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class GroupQueryAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int,
                 *, rngs: nnx.Rngs):
        assert num_heads % num_kv_heads == 0, "num_heads must divide by num_kv_heads"
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads

        self.W_q = nnx.Linear(d_model, d_model, rngs=rngs)
        # Note the smaller output width — this is where the saving comes from.
        self.W_k = nnx.Linear(d_model, num_kv_heads * self.d_k, rngs=rngs)
        self.W_v = nnx.Linear(d_model, num_kv_heads * self.d_k, rngs=rngs)
        self.W_o = nnx.Linear(d_model, d_model, rngs=rngs)

    def __call__(self, x):
        B, seq, _ = x.shape

        q = self.W_q(x).reshape(B, seq, self.num_heads, self.d_k).transpose(0, 2, 1, 3)
        k = self.W_k(x).reshape(B, seq, self.num_kv_heads, self.d_k).transpose(0, 2, 1, 3)
        v = self.W_v(x).reshape(B, seq, self.num_kv_heads, self.d_k).transpose(0, 2, 1, 3)

        # repeat, NOT tile: [a, b] -> [a, a, b, b] so query group g maps to
        # KV head g // repeats.
        repeats = self.num_heads // self.num_kv_heads
        k = jnp.repeat(k, repeats, axis=1)
        v = jnp.repeat(v, repeats, axis=1)

        # == q @ jnp.swapaxes(k, -1, -2)
        scores = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(
            jnp.asarray(self.d_k, x.dtype)
        )
        weights = jax.nn.softmax(scores, axis=-1)
        attn = jnp.einsum("bhqk,bhkd->bhqd", weights, v)      # == weights @ v

        out = attn.transpose(0, 2, 1, 3).reshape(B, seq, -1)
        return self.W_o(out)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

x = jax.random.normal(jax.random.key(1), (1, 8, 64))
for kv in (8, 4, 1):
    m = GroupQueryAttention(64, 8, kv, rngs=nnx.Rngs(params=0))
    label = {8: "MHA", 1: "MQA"}.get(kv, "GQA")
    print(f"  num_kv_heads={kv} ({label:3s})  out {m(x).shape}  "
          f"KV cache per token: {2 * kv * 8} values")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("gqa")